# 채소 종류·신선도 분류 결과 (10개 클래스)

최종 모델은 `model.pth`(EfficientNet-B0 미세조정, 오이·감자·토마토·피망·당근 × 정상·비정상)이다. 데이터는 1,860장으로 Train 1,260 / Valid 300 / Test 300장이며, 정상:비정상은 모든 분할에서 5:5다. 사진은 `dataset/` 아래에 분할(train·valid·test)과 상태(P_fresh 정상·N_rotten 비정상)별 폴더로 나눠 두었고, 데이터 설명은 `dataset/README.md`에 있다. 이 노트북은 README가 있는 폴더에서 실행한다. 그래프를 보려면 matplotlib이 필요하다.

In [ ]:
from pathlib import Path
import json, csv
import sys
import torch
sys.path.insert(0, "src")
from model_utils import load_model, read_image, CLASS_NAMES
ROOT = Path.cwd()
assert (ROOT / "model.pth").is_file(), "model.pth가 있는 폴더에서 실행하세요."
model, checkpoint = load_model(ROOT / "model.pth", torch.device("cpu"))
print("구조:", checkpoint["architecture"], "/ 학습 방식:", checkpoint["training_mode"])
print("선택된 epoch:", checkpoint["best_epoch"], "/", checkpoint["epochs_completed"])
print(json.dumps(checkpoint["counts"], indent=2, ensure_ascii=False))

## 모델별 Test 점수

`results/10class/`는 현재 10개 클래스 모델(Test 300장), `results/` 바로 아래는 6개 클래스 때의 비교 실험(같은 Test 180장)이다. Valid로만 epoch를 골랐고, Test는 마지막에 한 번만 채점했다.

In [ ]:
for run in sorted((ROOT / "results").iterdir()) + sorted((ROOT / "results" / "10class").iterdir()):
    f = run / "metrics.json"
    if not f.is_file():
        continue
    m = json.loads(f.read_text(encoding="utf-8"))["metrics"]
    if "test" not in m:
        continue
    t = m["test"]
    name = f"{run.parent.name}/{run.name}" if run.parent.name == "10class" else run.name
    print(f"{name:36s} {len(t['class_names'])}클래스 {t['accuracy']:.3f}  종류 {t['species_accuracy']:.3f}  정상/비정상 {t['condition_accuracy']:.3f}  (Test {t['sample_count']}장)")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
names = ["results/10class/efficientnet_b0_finetune/learning_curves.png", "results/10class/efficientnet_b0_finetune/confusion_matrix_test.png", "results/model_comparison.png"]
fig, axes = plt.subplots(len(names), 1, figsize=(12, 15))
for ax, name in zip(axes, names):
    ax.imshow(Image.open(ROOT / name))
    ax.axis("off")
plt.show()

## 틀린 Test 사진 보기

In [ ]:
rows = list(csv.DictReader(open(ROOT / "results/10class/efficientnet_b0_finetune/test_predictions.csv", encoding="utf-8")))
wrong = [r for r in rows if r["correct"] == "False"]
print(len(wrong), "장 틀림")
fig, axes = plt.subplots(1, max(1, min(6, len(wrong))), figsize=(18, 3.5), squeeze=False)
for ax, r in zip(axes[0], wrong[:6]):
    ax.imshow(Image.open(ROOT / "dataset" / r["image_path"]).convert("RGB"))
    ax.set_title(f"정답 {r['true_class']}\n예측 {r['predicted_class']}", fontsize=8)
    ax.axis("off")
plt.show()

## 직접 찍은 사진 예측

`my_photos` 폴더를 만들고 사진을 넣은 다음 `PHOTO`를 바꾼다. 점수가 0.5보다 낮으면 판별이 불확실하다는 뜻이다. 오이·감자·토마토·피망·당근이 아닌 사진도 10개 중 하나로 답하므로 주의한다.

In [ ]:
PHOTO = ROOT / "my_photos" / "example.jpg"
if PHOTO.is_file():
    with torch.inference_mode():
        probabilities = model(read_image(PHOTO).unsqueeze(0)).softmax(1)[0]
    index = int(probabilities.argmax())
    print("예측:", CLASS_NAMES[index], "/ 점수:", round(float(probabilities[index]), 3), "(불확실)" if probabilities[index] < 0.5 else "")
    plt.imshow(Image.open(PHOTO)); plt.axis("off"); plt.show()
else:
    print("my_photos에 사진을 넣고 PHOTO 경로를 수정하세요.")

## 다시 학습하기

터미널에서 실행한다.

```bash
python src/train.py --manifest dataset/manifest.csv --out results/10class/efficientnet_b0_finetune --arch efficientnet_b0 --mode finetune --epochs 30 --patience 8
```